<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 4: Implementing a GPT model from Scratch To Generate Text 

In [ ]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

- In this chapter, we implement a GPT-like LLM architecture; the next chapter will focus on training this LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/01.webp" width="500px">

&nbsp;
## 4.1 Coding an LLM architecture

- Chapter 1 discussed models like GPT and Llama, which generate words sequentially and are based on the decoder part of the original transformer architecture
- Therefore, these LLMs are often referred to as "decoder-like" LLMs
- Compared to conventional deep learning models, LLMs are larger, mainly due to their vast number of parameters, not the amount of code
- We'll see that many elements are repeated in an LLM's architecture

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/02.webp" width="400px">

- In previous chapters, we used small embedding dimensions for token inputs and outputs for ease of illustration, ensuring they fit on a single page
- In this chapter, we consider embedding and model sizes akin to a small GPT-2 model
- We'll specifically code the architecture of the smallest GPT-2 model (124 million parameters), as outlined in Radford et al.'s [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (note that the initial report lists it as 117M parameters, but this was later corrected in the model weight repository)
- Chapter 6 will show how to load pretrained weights into our implementation, which will be compatible with model sizes of 345, 762, and 1542 million parameters

- Configuration details for the 124 million parameter GPT-2 model include:

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

- We use short variable names to avoid long lines of code later
- `"vocab_size"` indicates a vocabulary size of 50,257 words, supported by the BPE tokenizer discussed in Chapter 2
- `"context_length"` represents the model's maximum input token count, as enabled by positional embeddings covered in Chapter 2
- `"emb_dim"` is the embedding size for token inputs, converting each input token into a 768-dimensional vector
- `"n_heads"` is the number of attention heads in the multi-head attention mechanism implemented in Chapter 3
- `"n_layers"` is the number of transformer blocks within the model, which we'll implement in upcoming sections
- `"drop_rate"` is the dropout mechanism's intensity, discussed in Chapter 3; 0.1 means dropping 10% of hidden units during training to mitigate overfitting
- `"qkv_bias"` decides if the `Linear` layers in the multi-head attention mechanism (from Chapter 3) should include a bias vector when computing query (Q), key (K), and value (V) tensors; we'll disable this option, which is standard practice in modern LLMs; however, we'll revisit this later when loading pretrained GPT-2 weights from OpenAI into our reimplementation in chapter 5

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/03.webp" width="500px">

In [ ]:
import torch
import torch.nn as nn


# YOUR CODE HERE
#
# Build three classes that together form the skeleton of a GPT model. The
# two "Dummy" submodules are placeholders -- their forward methods return
# their input unchanged. The point of the dummies is to verify that
# tensors of the right shape flow through the entire pipeline before you
# implement the real LayerNorm and TransformerBlock later.
#
# 1. DummyGPTModel(nn.Module)
#
#    In __init__, take a `cfg` dictionary and create these submodules as
#    attributes on self:
#      - a token embedding layer mapping vocab_size token ids to emb_dim
#        vectors
#      - a position embedding layer mapping context_length positions to
#        emb_dim vectors
#      - a dropout layer using the configured drop_rate
#      - an nn.Sequential containing n_layers copies of DummyTransformerBlock
#      - a DummyLayerNorm with width emb_dim (this will be the final norm
#        before the LM head)
#      - an output Linear layer mapping emb_dim features to vocab_size
#        logits, with bias disabled (GPT-2 convention)
#
#    In forward(in_idx), where in_idx has shape (batch, seq_len):
#      - read seq_len off the input shape
#      - look up token embeddings from in_idx, and positional embeddings
#        from torch.arange(seq_len, device=in_idx.device)
#      - add the two embedding tensors elementwise to form the input
#        representation
#      - pass that through dropout, then the transformer-block sequence,
#        then the final layer norm, then the output linear layer
#      - return the resulting logits tensor of shape
#        (batch, seq_len, vocab_size)
#
# 2. DummyTransformerBlock(nn.Module)
#    A no-op stand-in. The constructor takes cfg but doesn't need to use
#    it. The forward method receives x and returns it unchanged.
#
# 3. DummyLayerNorm(nn.Module)
#    Same idea -- the constructor takes normalized_shape and eps (default
#    1e-5) but does nothing with them. The forward method returns its
#    input unchanged.


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, in_idx):
        # YOUR CODE HERE
        pass


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        # YOUR CODE HERE
        pass


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        # YOUR CODE HERE
        pass


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/04.webp?123" width="500px">

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

---

**Note**

- If you are running this code on Windows or Linux, the resulting values above may look like as follows:
    
```
Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.9289,  0.2748, -0.7557,  ..., -1.6070,  0.2702, -0.5888],
         [-0.4476,  0.1726,  0.5354,  ..., -0.3932,  1.5285,  0.8557],
         [ 0.5680,  1.6053, -0.2155,  ...,  1.1624,  0.1380,  0.7425],
         [ 0.0447,  2.4787, -0.8843,  ...,  1.3219, -0.0864, -0.5856]],

        [[-1.5474, -0.0542, -1.0571,  ..., -1.8061, -0.4494, -0.6747],
         [-0.8422,  0.8243, -0.1098,  ..., -0.1434,  0.2079,  1.2046],
         [ 0.1355,  1.1858, -0.1453,  ...,  0.0869, -0.1590,  0.1552],
         [ 0.1666, -0.8138,  0.2307,  ...,  2.5035, -0.3055, -0.3083]]],
       grad_fn=<UnsafeViewBackward0>)
```

- Since these are just random numbers, this is not a reason for concern, and you can proceed with the remainder of the chapter without issues
- One possible reason for this discrepancy is the differing behavior of `nn.Dropout` across operating systems, depending on how PyTorch was compiled, as discussed [here on the PyTorch issue tracker](https://github.com/pytorch/pytorch/issues/121595)

---

&nbsp;
## 4.2 Normalizing activations with layer normalization

- Layer normalization, also known as LayerNorm ([Ba et al. 2016](https://arxiv.org/abs/1607.06450)), centers the activations of a neural network layer around a mean of 0 and normalizes their variance to 1
- This stabilizes training and enables faster convergence to effective weights
- Layer normalization is applied both before and after the multi-head attention module within the transformer block, which we will implement later; it's also applied before the final output layer

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/05.webp" width="400px">

- Let's see how layer normalization works by passing a small input sample through a simple neural network layer:

In [ ]:
torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5) 

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

- Let's compute the mean and variance for each of the 2 inputs above:

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

- The normalization is applied to each of the two inputs (rows) independently; using dim=-1 applies the calculation across the last dimension (in this case, the feature dimension) instead of the row dimension

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/06.webp" width="400px">

- Subtracting the mean and dividing by the square-root of the variance (standard deviation) centers the inputs to have a mean of 0 and a variance of 1 across the column (feature) dimension:

In [ ]:
# YOUR CODE HERE
#
# 1. Compute the normalised output by subtracting the row-wise mean from
#    `out` (the layer output from the previous cell) and dividing the
#    result by the square root of the row-wise variance. Store it as
#    `out_norm`.
#
# 2. Recompute the mean and variance of `out_norm` along the last
#    dimension (keeping that dim as size 1) to verify that the per-row
#    mean is now (close to) zero and the per-row variance is (close to)
#    one. Reassign these to `mean` and `var`.
#
# 3. Print out_norm, then mean, then var.


- Each input is centered at 0 and has a unit variance of 1; to improve readability, we can disable PyTorch's scientific notation:

In [ ]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

- Above, we normalized the features of each input
- Now, using the same idea, we can implement a `LayerNorm` class:

In [ ]:
class LayerNorm(nn.Module):
    """Per-token layer normalisation with trainable scale and shift parameters.

    Normalises each row (each token's feature vector) independently by
    subtracting the row mean and dividing by the row standard deviation,
    then applies a learnable per-feature scale and shift.
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        # YOUR CODE HERE
        #
        # Create two trainable parameters as attributes on self:
        #   - `scale` -- an nn.Parameter initialised to all ones, of shape
        #     (emb_dim,). This is the per-feature multiplicative scaling.
        #   - `shift` -- an nn.Parameter initialised to all zeros, of shape
        #     (emb_dim,). This is the per-feature additive offset.

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Compute the row-wise mean and the row-wise variance along the
        # last dimension, keeping that dim as size 1 (so the result
        # broadcasts back against x). Use the BIASED variance (no
        # Bessel correction) to match the canonical LayerNorm definition.
        #
        # Compute the normalised tensor by subtracting the mean and
        # dividing by the square root of (variance + self.eps). Adding
        # eps before the sqrt avoids a division-by-zero when an entire
        # row happens to be constant.
        #
        # Apply the trainable affine transform: multiply by self.scale
        # and add self.shift. Return the result.
        pass


**Scale and shift**

- Note that in addition to performing the normalization by subtracting the mean and dividing by the variance, we added two trainable parameters, a `scale` and a `shift` parameter
- The initial `scale` (multiplying by 1) and `shift` (adding 0) values don't have any effect; however, `scale` and `shift` are trainable parameters that the LLM automatically adjusts during training if it is determined that doing so would improve the model's performance on its training task
- This allows the model to learn appropriate scaling and shifting that best suit the data it is processing
- Note that we also add a smaller value (`eps`) before computing the square root of the variance; this is to avoid division-by-zero errors if the variance is 0

**Biased variance**
- In the variance calculation above, setting `unbiased=False` means using the formula $\frac{\sum_i (x_i - \bar{x})^2}{n}$ to compute the variance where n is the sample size (here, the number of features or columns); this formula does not include Bessel's correction (which uses `n-1` in the denominator), thus providing a biased estimate of the variance 
- For LLMs, where the embedding dimension `n` is very large, the difference between using n and `n-1`
 is negligible
- However, GPT-2 was trained with a biased variance in the normalization layers, which is why we also adopted this setting for compatibility reasons with the pretrained weights that we will load in later chapters

- Let's now try out `LayerNorm` in practice:

In [ ]:
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)

In [ ]:
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

- Variance is not exactly 1 because we use `eps`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/07.webp" width="400px">

&nbsp;
## 4.3 Implementing a feed forward network with GELU activations

- In this section, we implement a small neural network submodule that is used as part of the transformer block in LLMs
- We start with the activation function
- In deep learning, ReLU (Rectified Linear Unit) activation functions are commonly used due to their simplicity and effectiveness in various neural network architectures
- In LLMs, various other types of activation functions are used beyond the traditional ReLU; two notable examples are GELU (Gaussian Error Linear Unit) and SwiGLU (Swish-Gated Linear Unit)
- GELU and SwiGLU are more complex, smooth activation functions incorporating Gaussian and sigmoid-gated linear units, respectively, offering better performance for deep learning models, unlike the simpler, piecewise linear function of ReLU

- GELU ([Hendrycks and Gimpel 2016](https://arxiv.org/abs/1606.08415)) can be implemented in several ways; the exact version is defined as GELU(x)=x⋅Φ(x), where Φ(x) is the cumulative distribution function of the standard Gaussian distribution.
- In practice, it's common to implement a computationally cheaper approximation: $\text{GELU}(x) \approx 0.5 \cdot x \cdot \left(1 + \tanh\left[\sqrt{\frac{2}{\pi}} \cdot \left(x + 0.044715 \cdot x^3\right)\right]\right)
$ (the original GPT-2 model was also trained with this approximation)

In [ ]:
class GELU(nn.Module):
    """The GELU activation function used by GPT-2, computed via the
    tanh-based approximation.

    Mathematical definition:

        GELU(x) = 0.5 * x * (1 + tanh( sqrt(2/pi) * (x + 0.044715 * x^3) ))
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Implement the tanh-based GELU approximation exactly as written
        # in the formula above, using PyTorch tensor operations so it
        # broadcasts element-wise across any input shape. Use torch.tanh,
        # torch.sqrt, torch.tensor for the constant under the square root,
        # torch.pow for the cubic term, and torch.pi for the constant pi.
        # Return the resulting tensor (no parameters to update -- this
        # module is pure math).
        pass


In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# Some sample data
x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

- As we can see, ReLU is a piecewise linear function that outputs the input directly if it is positive; otherwise, it outputs zero
- GELU is a smooth, non-linear function that approximates ReLU but with a non-zero gradient for negative values (except at approximately -0.75)

- Next, let's implement the small neural network module, `FeedForward`, that we will be using in the LLM's transformer block later:

In [ ]:
class FeedForward(nn.Module):
    """The position-wise feed-forward network used inside each transformer
    block. It expands the embedding dimension by 4x, applies GELU, then
    projects back down to the original embedding dimension.

    Layout: Linear(emb_dim -> 4*emb_dim) -> GELU -> Linear(4*emb_dim -> emb_dim)
    """
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE HERE
        #
        # Create one attribute on self called `layers`, an nn.Sequential
        # containing three modules in order:
        #   - a Linear that maps emb_dim features to 4*emb_dim features
        #   - a GELU activation (the class you just implemented)
        #   - a Linear that maps 4*emb_dim features back to emb_dim features
        # Read emb_dim from the cfg dictionary.
        pass

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Pass x through self.layers and return the result.
        pass


In [ ]:
print(GPT_CONFIG_124M["emb_dim"])

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/09.webp?12" width="400px">

In [ ]:
ffn = FeedForward(GPT_CONFIG_124M)

# input shape: [batch_size, num_token, emb_size]
x = torch.rand(2, 3, 768) 
out = ffn(x)
print(out.shape)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/10.webp" width="400px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/11.webp" width="400px">

&nbsp;
## 4.4 Adding shortcut connections

- Next, let's talk about the concept behind shortcut connections, also called skip or residual connections
- Originally, shortcut connections were proposed in deep networks for computer vision (residual networks) to mitigate vanishing gradient problems
- A shortcut connection creates an alternative shorter path for the gradient to flow through the network
- This is achieved by adding the output of one layer to the output of a later layer, usually skipping one or more layers in between
- Let's illustrate this idea with a small example network:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/12.webp?123" width="400px">

- In code, it looks like this:

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    """A 5-layer fully-connected network with an OPTIONAL residual
    (shortcut) connection on each layer. We use this purely to
    demonstrate the vanishing-gradient problem when shortcuts are absent
    and how residual connections fix it.

    The constructor takes a list of six integers describing the input,
    four hidden, and output widths -- so the network has five Linear
    layers wiring them together, each followed by GELU. It also takes a
    `use_shortcut` boolean that toggles whether residuals are applied
    in the forward pass.

    In the forward pass, x is passed through every layer in turn. After
    each layer, if `use_shortcut` is True AND the layer's input and
    output have the same shape, the layer's output is ADDED to the
    layer's input (a residual connection). Otherwise the output simply
    replaces x for the next iteration.
    """
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        # YOUR CODE HERE
        #
        # Create one attribute on self called `layers`, an nn.ModuleList
        # of five nn.Sequential blocks. Each block contains:
        #   - a Linear mapping layer_sizes[i] features to layer_sizes[i+1]
        #     features
        #   - a GELU activation
        # The ModuleList has one block per consecutive pair in layer_sizes
        # (i.e. (0,1), (1,2), (2,3), (3,4), (4,5)).

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Loop over self.layers. For each layer:
        #   - Compute the layer's output by calling it on x.
        #   - If self.use_shortcut is True AND the input x has the same
        #     shape as the layer output, set x to the sum of the two
        #     (the residual). Otherwise set x to just the layer output.
        # After the loop, return x.
        pass


def print_gradients(model, x):
    """Run a forward + backward pass through `model` and print the mean
    of the absolute gradient of every weight parameter. This shows how
    much gradient signal reaches each layer.
    """
    # YOUR CODE HERE
    #
    # 1. Call the model on x to get its output.
    # 2. Build a small target tensor of shape (1, 1) holding 0.0.
    # 3. Compute the mean-squared-error loss between the output and target.
    # 4. Call .backward() on the loss so gradients populate.
    # 5. Iterate model.named_parameters(). For each parameter whose name
    #    contains the substring "weight" (so biases are skipped), print
    #    the parameter name and the mean of the absolute value of its
    #    gradient.
    pass


- Let's print the gradient values first **without** shortcut connections:

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

- Next, let's print the gradient values **with** shortcut connections:

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

- As we can see based on the output above, shortcut connections prevent the gradients from vanishing in the early layers (towards `layer.0`)
- We will use this concept of a shortcut connection next when we implement a transformer block

&nbsp;
## 4.5 Connecting attention and linear layers in a transformer block

- In this section, we now combine the previous concepts into a so-called transformer block
- A transformer block combines the causal multi-head attention module from the previous chapter with the linear layers, the feed forward neural network we implemented in an earlier section
- In addition, the transformer block also uses dropout and shortcut connections

In [ ]:
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch03 import MultiHeadAttention

from previous_chapters import MultiHeadAttention


class TransformerBlock(nn.Module):
    """One transformer block in the pre-LayerNorm (GPT-2) style.

    Two sub-blocks, each wrapped in a residual connection. The first
    sub-block applies layer norm, then multi-head attention, then
    dropout, then adds the result back to the original input. The
    second sub-block does the same with the FFN: layer norm, then the
    feed-forward network, then dropout, then a residual add.

    The "pre-LN" name means the LayerNorm is applied BEFORE the
    attention/FFN, not after.
    """
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE HERE
        #
        # Create five submodules as attributes on self, reading sizes
        # from the cfg dictionary:
        #   - att: a MultiHeadAttention with d_in=emb_dim, d_out=emb_dim,
        #     context_length=context_length, num_heads=n_heads,
        #     dropout=drop_rate, and qkv_bias=qkv_bias
        #   - ff: a FeedForward built from the cfg
        #   - norm1: a LayerNorm with width emb_dim (used before attention)
        #   - norm2: a LayerNorm with width emb_dim (used before the FFN)
        #   - drop_shortcut: an nn.Dropout with probability drop_rate
        #     (applied to the output of each sub-block before the residual
        #     add)
        pass

    def forward(self, x):
        # YOUR CODE HERE
        #
        # Implement the two pre-LN residual sub-blocks.
        #
        # First sub-block (attention):
        #   - Save the original x as a shortcut copy.
        #   - Apply norm1, then attention, then drop_shortcut, in that
        #     order, overwriting x at each step.
        #   - Add the shortcut copy back into x (the residual connection).
        #
        # Second sub-block (FFN):
        #   - Save the now-updated x as a new shortcut.
        #   - Apply norm2, then the feed-forward network, then
        #     drop_shortcut.
        #   - Add the shortcut back in.
        #
        # Return the final x. The output shape matches the input shape.
        pass


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/13.webp?1" width="400px">

- Suppose we have 2 input samples with 4 tokens each, where each token is a 768-dimensional embedding vector; then this transformer block applies self-attention, followed by linear layers, to produce an output of similar size
- You can think of the output as an augmented version of the context vectors we discussed in the previous chapter

In [ ]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/14.webp?1" width="400px">

&nbsp;
## 4.6 Coding the GPT model

- We are almost there: now let's plug in the transformer block into the architecture we coded at the very beginning of this chapter so that we obtain a usable GPT architecture
- Note that the transformer block is repeated multiple times; in the case of the smallest 124M GPT-2 model, we repeat it 12 times:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/15.webp" width="400px">

- The corresponding code implementation, where `cfg["n_layers"] = 12`:

In [ ]:
class GPTModel(nn.Module):
    """The full GPT model: token + positional embeddings, n_layers stacked
    transformer blocks, a final LayerNorm, and an output linear head
    (the LM head) projecting back to vocab_size logits.

    The output has shape (batch, seq_len, vocab_size) -- one logit
    vector per input position, predicting the distribution over the
    next token.
    """
    def __init__(self, cfg):
        super().__init__()
        # YOUR CODE HERE
        #
        # Create six submodules as attributes on self, reading sizes from
        # the cfg dictionary:
        #   - tok_emb: an nn.Embedding mapping vocab_size token ids to
        #     emb_dim vectors
        #   - pos_emb: an nn.Embedding mapping context_length positions
        #     to emb_dim vectors
        #   - drop_emb: an nn.Dropout with probability drop_rate (applied
        #     to the input embeddings)
        #   - trf_blocks: an nn.Sequential containing n_layers copies of
        #     TransformerBlock, each built from the same cfg
        #   - final_norm: a LayerNorm with width emb_dim (applied after
        #     the last transformer block)
        #   - out_head: a Linear mapping emb_dim features to vocab_size
        #     logits, with bias=False (GPT-2 convention)
        pass

    def forward(self, in_idx):
        # YOUR CODE HERE
        #
        # in_idx has shape (batch_size, seq_len) and holds integer token ids.
        #
        # Read seq_len off the input shape.
        #
        # Build the input representation by ADDING two embeddings:
        #   - token embeddings looked up from in_idx
        #   - positional embeddings looked up from a 0..seq_len-1 arange
        #     tensor on the same device as in_idx
        #
        # Apply drop_emb to the summed embeddings, then pass the result
        # through trf_blocks (the stack of transformer blocks), then
        # through final_norm, then through out_head to produce logits.
        # Return the logits tensor of shape (batch, seq_len, vocab_size).
        pass


- Using the configuration of the 124M parameter model, we can now instantiate this GPT model with random initial weights as follows:

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

- We will train this model in the next chapter
- However, a quick note about its size: we previously referred to it as a 124M parameter model; we can double check this number as follows:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

- As we see above, this model has 163M, not 124M parameters; why?
- In the original GPT-2 paper, the researchers applied weight tying, which means that they reused the token embedding layer (`tok_emb`) as the output layer, which means setting `self.out_head.weight = self.tok_emb.weight`
- The token embedding layer projects the 50,257-dimensional one-hot encoded input tokens to a 768-dimensional embedding representation
- The output layer projects 768-dimensional embeddings back into a 50,257-dimensional representation so that we can convert these back into words (more about that in the next section)
- So, the embedding and output layer have the same number of weight parameters, as we can see based on the shape of their weight matrices
- However, a quick note about its size: we previously referred to it as a 124M parameter model; we can double check this number as follows:

In [ ]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

- In the original GPT-2 paper, the researchers reused the token embedding matrix as an output matrix
- Correspondingly, if we subtracted the number of parameters of the output layer, we'd get a 124M parameter model:

In [ ]:
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

- In practice, I found it easier to train the model without weight-tying, which is why we didn't implement it here
- However, we will revisit and apply this weight-tying idea later when we load the pretrained weights in chapter 5
- Lastly, we can compute the memory requirements of the model as follows, which can be a helpful reference point:

In [ ]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

- Exercise: you can try the following other configurations, which are referenced in the [GPT-2 paper](https://scholar.google.com/citations?view_op=view_citation&hl=en&user=dOad5HoAAAAJ&citation_for_view=dOad5HoAAAAJ:YsMSGLbcyi4C), as well.

    - **GPT2-small** (the 124M configuration we already implemented):
        - "emb_dim" = 768
        - "n_layers" = 12
        - "n_heads" = 12

    - **GPT2-medium:**
        - "emb_dim" = 1024
        - "n_layers" = 24
        - "n_heads" = 16
    
    - **GPT2-large:**
        - "emb_dim" = 1280
        - "n_layers" = 36
        - "n_heads" = 20
    
    - **GPT2-XL:**
        - "emb_dim" = 1600
        - "n_layers" = 48
        - "n_heads" = 25

&nbsp;
## 4.7 Generating text

- LLMs like the GPT model we implemented above are used to generate one word at a time

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/16.webp" width="400px">

- The following `generate_text_simple` function implements greedy decoding, which is a simple and fast method to generate text
- In greedy decoding, at each step, the model chooses the word (or token) with the highest probability as its next output (the highest logit corresponds to the highest probability, so we technically wouldn't even have to compute the softmax function explicitly)
- In the next chapter, we will implement a more advanced `generate_text` function
- The figure below depicts how the GPT model, given an input context, generates the next word token

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/17.webp" width="600px">

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """Greedy autoregressive text generation.

    The argument `idx` is a tensor of shape (batch, n_tokens) holding the
    starting context as token ids. At each step the function appends one
    new token (the argmax of the model's predicted next-token
    distribution) to idx, runs for max_new_tokens steps, and returns the
    extended idx tensor.
    """
    # YOUR CODE HERE
    #
    # Loop for max_new_tokens iterations. In each iteration:
    #
    #   1. Crop the running context to the most recent `context_size`
    #      tokens. (The model can only attend up to context_size
    #      positions, so longer contexts must be trimmed from the left.)
    #
    #   2. Run the cropped context through the model inside a no-grad
    #      context to save memory and skip autograd bookkeeping. The
    #      result is a logits tensor of shape (batch, seq_len, vocab_size).
    #
    #   3. Keep only the logits at the LAST position along the sequence
    #      axis -- these are the predictions for the next token. The
    #      shape becomes (batch, vocab_size).
    #
    #   4. Convert the last-position logits into a probability
    #      distribution by applying softmax along the vocabulary axis.
    #
    #   5. Take the argmax along the vocabulary axis to pick the most
    #      likely next token id. Keep the resulting shape as (batch, 1)
    #      using keepdim=True so the next step can concatenate cleanly.
    #
    #   6. Concatenate the new (batch, 1) token onto idx along the
    #      sequence axis, growing idx from (batch, n) to (batch, n+1).
    #
    # After the loop, return idx.
    for _ in range(max_new_tokens):
        pass
    return idx


- The `generate_text_simple` above implements an iterative process, where it creates one token at a time

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch04_compressed/18.webp" width="600px">

- Let's prepare an input example:

In [ ]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

- Remove batch dimension and convert back into text:

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

- Note that the model is untrained; hence the random output texts above
- We will train the model in the next chapter

&nbsp;
## Summary and takeaways

- See the [./gpt.py](./gpt.py) script, a self-contained script containing the GPT model we implement in this Jupyter notebook
- You can find the exercise solutions in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)